# MPC Upstream With First-Step Lyapunov Contraction

This notebook mirrors the safety MPC notebook architecture: same refined selector, same effective-target reuse, same debug/export flow, but no QCQP projection. The only Lyapunov enforcement is a hard first-step contraction constraint added to the upstream MPC solve.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from Simulation.system_functions import PolymerCSTR
from Simulation.mpc import MpcSolver, compute_observer_gain
from Simulation.run_mpc_first_step_contraction import run_mpc_first_step_contraction
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from utils.td3_helpers import load_and_prepare_system_data
from utils.scaling_helpers import apply_min_max
from Lyapunov.safety_debug import build_safety_filter_run_bundle, save_safety_filter_debug_artifacts


In [ ]:
# Polymer CSTR setup
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_phys = np.array([[4.5, 324.0], [3.4, 321.0]])

system_dict_path = os.path.join("Data", "system_dict")
augmentation_style = "rawlings"
augmentation_mode = "mixed_B_I"

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_phys,
    u_min=u_min,
    u_max=u_max,
    system_dict_path=system_dict_path,
    augmentation_style=augmentation_style,
    augmentation_mode=augmentation_mode,
)

A = system_data["A"]
B = system_data["B"]
C = system_data["C"]
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
Bd_used = system_data["Bd_used"]
Cd_used = system_data["Cd_used"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False]

predict_h = 9
cont_h = 3
warm_start = 0

u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
b_min = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
b_max = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])
b1 = (b_min[0] - u_ss[0], b_max[0] - u_ss[0])
b2 = (b_min[1] - u_ss[1], b_max[1] - u_ss[1])
bnds = (b1, b2) * cont_h
cons = []
IC_opt = np.zeros(inputs_number * cont_h)

Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

Qy_diag_std = np.array([Q1_penalty, Q2_penalty], dtype=float)
Su_diag_std = np.array([0.0, 0.0], dtype=float)
Rdu_diag_std = np.array([R1_penalty, R2_penalty], dtype=float)

observer_poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                           0.42900673, 0.4228262, 0.96916776, 0.91230187])
L = compute_observer_gain(A_aug, C_aug, observer_poles)

reward_config, reward_fn = make_reward_fn_mpc_quadratic(
    Q_diag=np.array([Q1_penalty, Q2_penalty]),
    R_diag=np.array([R1_penalty, R2_penalty]),
)

MPC_obj = MpcSolver(
    A_aug,
    B_aug,
    C_aug,
    Q_out=Qy_diag_std,
    R_in=Rdu_diag_std,
    NP=predict_h,
    NC=cont_h,
)

selector_config = {
    "alpha_u_ref": 0.5,
    "alpha_du_sel": 0.5,
    "alpha_dx_sel": 0.05,
    "alpha_x_ref": 0.01,
    "x_weight_base": "CtQC",
    "use_output_bounds_in_selector": True,
}

Qs_tgt_diag = 1e8 * np.asarray(MPC_obj.Q_out, float)
Ru_tgt_diag = 1.0 * np.ones(MPC_obj.B.shape[1])

run_config = {
    "rho_lyap": 0.98,
    "lyap_eps": 1e-9,
    "lyap_tol": 1e-10,
    "fallback_policy": "offset_free_mpc",
    "mpc_target_policy": "raw_setpoint",
    "tracking_target_policy": "raw_setpoint",
    "target_selector_config": selector_config,
    "selector_H": None,
    "target_backup_policy": "last_valid",
    "selector_warm_start": True,
    "reuse_mpc_solution_as_ic": False,
    "reset_system_on_entry": True,
    "first_step_contraction_on": True,
}

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92


In [ ]:
print("Using refined Step A selector with MPC upstream first-step contraction")

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)

results_first_step = run_mpc_first_step_contraction(
    system=cstr,
    MPC_obj=MPC_obj,
    y_sp_scenario=y_sp_scenario,
    n_tests=n_tests,
    set_points_len=set_points_len,
    steady_states=steady_states,
    IC_opt=IC_opt,
    bnds=bnds,
    cons=cons,
    warm_start=warm_start,
    L=L,
    data_min=data_min,
    data_max=data_max,
    test_cycle=TEST_CYCLE,
    reward_fn=reward_fn,
    nominal_qi=nominal_qi,
    nominal_qs=nominal_qs,
    nominal_ha=nominal_hA,
    qi_change=qi_change,
    qs_change=qs_change,
    ha_change=ha_change,
    mode="disturb",
    rho_lyap=run_config["rho_lyap"],
    lyap_eps=run_config["lyap_eps"],
    lyap_tol=run_config["lyap_tol"],
    Qy_track_diag=None,
    Rmove_diag=None,
    Qs_tgt_diag=Qs_tgt_diag,
    Ru_tgt_diag=Ru_tgt_diag,
    fallback_policy=run_config["fallback_policy"],
    target_selector_config=run_config["target_selector_config"],
    selector_H=run_config["selector_H"],
    mpc_target_policy=run_config["mpc_target_policy"],
    tracking_target_policy=run_config["tracking_target_policy"],
    target_backup_policy=run_config["target_backup_policy"],
    selector_warm_start=run_config["selector_warm_start"],
    reuse_mpc_solution_as_ic=run_config["reuse_mpc_solution_as_ic"],
    reset_system_on_entry=run_config["reset_system_on_entry"],
    first_step_contraction_on=run_config["first_step_contraction_on"],
)

bundle = build_safety_filter_run_bundle(
    source="mpc_first_step_contraction",
    results=results_first_step,
    steady_states=steady_states,
    config=run_config,
    min_max_dict=min_max_dict,
    data_min=data_min,
    data_max=data_max,
    extra={"reward_config": reward_config, "delta_t": delta_t},
)

debug_dir = save_safety_filter_debug_artifacts(
    bundle=bundle,
    directory=os.path.join(os.getcwd(), "Data", "debug_exports"),
    prefix_name="mpc_first_step_contraction",
    save_plots=True,
)

bundle["summary"], debug_dir


In [ ]:
(
    y_first,
    u_first,
    avg_rewards_first,
    rewards_first,
    xhatdhat_first,
    nFE_first,
    time_in_sub_episodes_first,
    y_sp_first,
    yhat_first,
    e_store_first,
    qi_first,
    qs_first,
    ha_first,
    lmpc_info_storage_first,
    u_safe_dev_store_first,
) = results_first_step

last_info = lmpc_info_storage_first[-1] if lmpc_info_storage_first else {}
summary = {
    "correction_mode": last_info.get("correction_mode"),
    "verified": last_info.get("verified"),
    "V_k": last_info.get("V_k"),
    "V_next_first": last_info.get("V_next_first"),
    "V_bound": last_info.get("V_bound"),
    "contraction_margin": last_info.get("contraction_margin"),
    "first_step_contraction_satisfied": last_info.get("first_step_contraction_satisfied"),
    "debug_dir": debug_dir,
}
summary


In [ ]:
print("Saved safety-style debug export to:", debug_dir)
print("Contraction plot:", os.path.join(debug_dir, "first_step_contraction_diagnostics.png"))


In [ ]:
time_u = np.arange(len(lmpc_info_storage_first)) * delta_t
V_k = np.array([np.nan if info.get("V_k") is None else float(info.get("V_k")) for info in lmpc_info_storage_first])
V_next_first = np.array([np.nan if info.get("V_next_first") is None else float(info.get("V_next_first")) for info in lmpc_info_storage_first])
V_bound = np.array([np.nan if info.get("V_bound") is None else float(info.get("V_bound")) for info in lmpc_info_storage_first])
contraction_margin = np.array([np.nan if info.get("contraction_margin") is None else float(info.get("contraction_margin")) for info in lmpc_info_storage_first])
contraction_ok = np.array([0.0 if not info.get("first_step_contraction_satisfied", False) else 1.0 for info in lmpc_info_storage_first])
target_success = np.array([0.0 if not info.get("target_success", False) else 1.0 for info in lmpc_info_storage_first])

plt.figure(figsize=(8.5, 8.0))
ax1 = plt.subplot(4, 1, 1)
ax1.plot(time_u, V_k, label="V_k", linewidth=2)
ax1.plot(time_u, V_next_first, label="V_next_first", linewidth=2)
ax1.plot(time_u, V_bound, label="V_bound", linestyle="--", linewidth=2)
ax1.set_ylabel("Lyapunov")
ax1.legend(loc="best")
ax1.grid(True, alpha=0.3)

ax2 = plt.subplot(4, 1, 2, sharex=ax1)
ax2.plot(time_u, contraction_margin, color="tab:red", linewidth=2)
ax2.axhline(0.0, color="k", linestyle="--", linewidth=1)
ax2.set_ylabel("margin")
ax2.grid(True, alpha=0.3)

ax3 = plt.subplot(4, 1, 3, sharex=ax1)
ax3.step(time_u, contraction_ok, where="post", linewidth=2, label="first-step contraction ok")
ax3.set_ylabel("ok")
ax3.set_ylim(-0.1, 1.1)
ax3.grid(True, alpha=0.3)
ax3.legend(loc="best")

ax4 = plt.subplot(4, 1, 4, sharex=ax1)
ax4.step(time_u, target_success, where="post", linewidth=2, label="target selector success")
ax4.set_ylabel("target")
ax4.set_xlabel("Time (h)")
ax4.set_ylim(-0.1, 1.1)
ax4.grid(True, alpha=0.3)
ax4.legend(loc="best")

plt.tight_layout()
plt.show()
